# ResNet-18 training on the augmented data

Trains the ResNet-18 classifier on one augmented dataset (augmentation version x content
variant) with the paper's trainer, then evaluates it. The last section reproduces the
paper's final cross-device result: the bundled final weights evaluated on the otoscope
test set (95.5% accuracy in the paper, on the full n = 90 set).

Training is run by the paper's own `train.py` (unchanged apart from English comments) --
one command per run, with per-epoch validation, best-model checkpointing, and early
stopping built in. This notebook calls it and evaluates the results with helpers in
`eval_lib.py`. The network is in `model.py` / `configs.py`, the loader in `dataset.py`.

`demo` mode does not train: the bundled sample is only a few images per class, too small
to train on. It runs section 4 -- the bundled final weight evaluated on the small
otoscope sample under `../demo_data/` -- so the classifier can be exercised end to end on
CPU without the full dataset. `real` mode runs the full train-and-evaluate path.

The full per-subset evaluation and the Figure-5 confusion-matrix panels are produced separately.

## 1. Mode and configuration

- **`demo`** -- no training. The bundled `../demo_data/` holds only a few images per
  class, so demo mode skips sections 2-3 and runs section 4: the bundled final weight
  evaluated on the small otoscope sample (CPU, seconds). This is a smoke test that the
  model loads and classifies end to end.
- **`real`** -- the paper run (up to 100 epochs, early stopping). Training reads the
  augmented dataset `<DATA_ROOT>/<DATASET>/`, produced by
  `../2_wct2_augmentation/augment.ipynb`. `DATA_ROOT` is `OTOSCOPE_DATA_ROOT` when set,
  otherwise the bundled `../real_data/`. A CUDA GPU is expected.

`DATASET` picks the augmented dataset; paper runs cover `{p, pb, sp, spb} x {raw, crop}`.
All eight paper runs use `TRIAL = 1`.

Section 4 (final weights on the otoscope test set) needs no training data and runs in
either mode.

In [ ]:
import os, sys, subprocess

MODE = "demo"                       # "demo" or "real"
DATASET = "dataset_spb_f2_crop"     # augmentation version x content variant
TRIAL = 1                           # all eight paper runs use trial 1

if MODE == "demo":
    # No training in demo -- only section 4 runs, on the bundled otoscope sample.
    DATA_ROOT = "../demo_data/classification"
    TRAIN_DIR = None
    EPOCHS = 0
    NUM_WORKERS = 0                 # load data in the main process
elif MODE == "real":
    DATA_ROOT = os.environ.get("OTOSCOPE_DATA_ROOT", "../real_data")
    TRAIN_DIR = os.path.join(DATA_ROOT, DATASET)
    EPOCHS = 100
    NUM_WORKERS = 4
else:
    raise ValueError("MODE must be 'demo' or 'real'")

CKPT_ROOT = "./__checkpoints"
LOG_ROOT  = "./__logs"

print(f"MODE={MODE}  DATASET={DATASET}  TRIAL={TRIAL}  EPOCHS={EPOCHS}")
print("DATA_ROOT:", DATA_ROOT)
if MODE == "demo":
    print("Demo mode: sections 2-3 (training) are skipped; run section 4.")
elif os.path.isdir(TRAIN_DIR):
    print("Training data found:", TRAIN_DIR)
else:
    print("Training data NOT found:", TRAIN_DIR)
    print("-> produce it with ../2_wct2_augmentation/augment.ipynb, or point")
    print("   OTOSCOPE_DATA_ROOT at a directory that contains it.")
    print("   (Section 4 below runs without it.)")

## 2. Train (real mode)

One `train.py` call per run -- the paper defaults (ResNet-18, dropout 0.3, Adam @ lr 5e-5,
weight decay 1e-4, batch 64, image size 256x256), per-epoch evaluation on the validation
sets, best-model checkpointing on the `Test` set loss, early stopping (patience 20),
metrics logging, and `--resume` for interrupted runs. Checkpoints go to
`./__checkpoints/<run>/`.

The validation sets are taken from `DATA_ROOT`: `roi_test` or `raw_test` matching the
content variant, plus `otoscope`. The `*_filter` variant is used when present and falls
back to the main set otherwise (it is logged only -- best-model selection uses `Test`).

This section is skipped in `demo` mode (see section 1); jump to section 4.

In [ ]:
if MODE == "demo":
    print("Demo mode: no training. Skip to section 4.")
else:
    if not os.path.isdir(TRAIN_DIR):
        raise FileNotFoundError(f"training data not found: {TRAIN_DIR} - see section 1")

    # Validation sets from DATA_ROOT; *_filter falls back to the main set when absent.
    variant  = "roi" if "crop" in DATASET else "raw"
    val_main = os.path.join(DATA_ROOT, "roi_test" if variant == "roi" else "raw_test")
    val_filt = val_main + "_filter" if os.path.isdir(val_main + "_filter") else val_main

    cmd = [
        sys.executable, "train.py",
        "--dataset", TRAIN_DIR, "--trial", str(TRIAL),
        "--epochs", str(EPOCHS), "--num-workers", str(NUM_WORKERS),
        "--val-mode", variant,
        f"--val-{variant}-dir",        val_main,
        f"--val-{variant}-filter-dir", val_filt,
        "--val-otoscope-dir",          os.path.join(DATA_ROOT, "otoscope"),
        "--ckpt-root", CKPT_ROOT, "--log-root", LOG_ROOT,
    ]
    run_name = f"{DATASET}_{TRIAL:02d}"

    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

    best_ckpt = os.path.join(CKPT_ROOT, run_name, "_ResNet_model_best.pt")
    print("Best checkpoint:", best_ckpt)

## 3. Evaluate the trained model (real mode)

Loads the best checkpoint from section 2 and reports top-1 / top-2 accuracy, a per-class
report, and a confusion matrix on the matching endoscope validation set. Skipped in
`demo` mode.

In [ ]:
if MODE == "demo":
    print("Demo mode: no trained checkpoint to evaluate. Skip to section 4.")
else:
    from eval_lib import (
        load_resnet, make_loader, predict, basic_metrics, plot_confusion, CLASS_NAMES,
    )

    EVAL_DIR = os.path.join(DATA_ROOT, "roi_test" if "crop" in DATASET else "raw_test")

    model  = load_resnet(best_ckpt, num_class=4, dropout=0.3)
    loader = make_loader(EVAL_DIR, num_workers=NUM_WORKERS)
    y_true, y_pred, probs = predict(model, loader)

    summary = basic_metrics(y_true, y_pred, probs, CLASS_NAMES)
    fig = plot_confusion(summary["confusion_matrix"], CLASS_NAMES,
                         title=f"{DATASET} - {os.path.basename(EVAL_DIR)}")
    fig.show()

## 4. Final model on the otoscope test set (no training required)

The paper's final configuration is S + C + EB -- `dataset_spb_f2_crop`, best epoch 37 --
bundled as `weights/dataset_spb_f2_crop_best.pt`. This section loads it and evaluates the
otoscope test set, reproducing the paper's headline cross-device result (95.5% accuracy on
the full n = 90 test set). It runs in either mode, over relative paths, independently of
sections 2-3:

- `demo` -- the small bundled sample under `../demo_data/classification/otoscope_sample/`
  (a few images per class); accuracy is only indicative at this size.
- `real` -- the full otoscope test set under `../real_data/otoscope` (n = 90), which
  reproduces the 95.5% figure.

To evaluate a different configuration, point `FINAL_WEIGHTS` at any of the eight files in
`weights/` (see `weights/README.md` for the file -> configuration -> accuracy mapping).

In [ ]:
from eval_lib import (   # repeated so this section runs without sections 2-3
    load_resnet, make_loader, predict, basic_metrics, plot_confusion, CLASS_NAMES,
)

FINAL_WEIGHTS = "./weights/dataset_spb_f2_crop_best.pt"   # the paper's final model (S + C + EB)
if MODE == "demo":
    OTOSCOPE_DIR = "../demo_data/classification/otoscope_sample"   # small bundled sample
else:
    OTOSCOPE_DIR = "../real_data/otoscope"                         # full otoscope test set (n = 90)

if not os.path.isfile(FINAL_WEIGHTS):
    raise FileNotFoundError(f"{FINAL_WEIGHTS} - see weights/README.md")
if not os.path.isdir(OTOSCOPE_DIR):
    raise FileNotFoundError(f"{OTOSCOPE_DIR} - otoscope test images required")

model  = load_resnet(FINAL_WEIGHTS, num_class=4, dropout=0.3)
loader = make_loader(OTOSCOPE_DIR, num_workers=NUM_WORKERS)
y_true, y_pred, probs = predict(model, loader)

summary = basic_metrics(y_true, y_pred, probs, CLASS_NAMES)
print(f"n = {len(y_true)} images  ({'demo sample' if MODE == 'demo' else 'full test set'})")
fig = plot_confusion(summary["confusion_matrix"], CLASS_NAMES,
                     title="Final model (spb_f2_crop) - otoscope test set")
fig.show()